# tm_final_xx

Placeholder notebook for the ready-to-run final solution.

This notebook will later contain the single production pipeline and final classifier required by the project handout.

In [2]:
# Agent logic stays in src; notebook only imports and uses it.
import os
import sys
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI

sys.path.insert(0, os.path.abspath("../src"))
from text_mining_utils import LABEL_MAP, preprocess_pipeline, detect_transformer_device
from agentic_tools import (
    build_agent_tools,
    create_specialist_executors,
    create_orchestrator_executor,
    create_orchestrator_memory_executor,
    print_trace,
 )

load_dotenv()
AZURE_MODEL_NAME = os.getenv("AZURE_MODEL_NAME", "")
AZURE_ENDPOINT = os.getenv("AZURE_ENDPOINT", "")
AZURE_KEY = os.getenv("AZURE_KEY", "")
AZURE_API_VERSION = os.getenv("AZURE_API_VERSION", "")

MAX_LENGTH = globals().get("MAX_LENGTH", 96)
device, device_name = detect_transformer_device()
transformer_runs = globals().get("transformer_runs", {})

# Keep references explicit for lint/analysis and notebook discoverability.
AGENT_UTILS = {
    "build_agent_tools": build_agent_tools,
    "create_specialist_executors": create_specialist_executors,
    "create_orchestrator_executor": create_orchestrator_executor,
    "create_orchestrator_memory_executor": create_orchestrator_memory_executor,
    "print_trace": print_trace,
}

print("Agent utilities loaded from src/agentic_tools.py")
print("Available:", ", ".join(AGENT_UTILS.keys()))

Agent utilities loaded from src/agentic_tools.py
Available: build_agent_tools, create_specialist_executors, create_orchestrator_executor, create_orchestrator_memory_executor, print_trace


# Agentic AI

We now expose a simplified agent workflow that uses the single best validated encoder-classifier model from the experiment results.

In [3]:
# Agent tools and model registry are now defined in src/agentic_tools.py
if not transformer_runs:
    raise RuntimeError(
        "transformer_runs is empty. Run training notebook cells first and pass/load transformer_runs here."
    )

tools_bundle = build_agent_tools(
    transformer_runs=transformer_runs,
    preprocess_pipeline_fn=preprocess_pipeline,
    label_map=LABEL_MAP,
    max_length=MAX_LENGTH,
    device=device,
)

MODEL_REGISTRY = tools_bundle["model_registry"]
MODEL_NAME_LOOKUP = tools_bundle["model_name_lookup"]
BEST_MODEL_NAME = tools_bundle["best_model_name"]

inspect_tweet_profile = tools_bundle["inspect_tweet_profile"]
get_validation_metrics = tools_bundle["get_validation_metrics"]
classify_with_best_model = tools_bundle["classify_with_best_model"]

RuntimeError: transformer_runs is empty. Run training notebook cells first and pass/load transformer_runs here.

In [ ]:
# Tools are loaded from src/agentic_tools.py and ready to use
print("Loaded tools:")
print("- inspect_tweet_profile")
print("- get_validation_metrics")
print("- classify_with_best_model")
print(f"Best model selected for agent: {BEST_MODEL_NAME}")

### Building the Conversational Agent

The structure below is modular: define tools, create specialized agents, then expose a single conversational interface through an orchestrator.


In [ ]:
# Initialise the Azure OpenAI LLM client using credentials loaded in cell 1
llm = AzureChatOpenAI(
    temperature=0,
    model=AZURE_MODEL_NAME,
    azure_endpoint=AZURE_ENDPOINT,
    openai_api_type="azure",
    api_key=AZURE_KEY,
    api_version=AZURE_API_VERSION,
)

print("LLM ready:", llm.model_name)


In [ ]:
# Create routing and classification specialist executors from src module
(
    routing_executor,
    classification_executor,
    routing_tools,
    classification_tools,
) = create_specialist_executors(llm, tools_bundle)

print("Routing tools:", [tool.name for tool in routing_tools])
print("Classification tools:", [tool.name for tool in classification_tools])

In [ ]:
# Create orchestrator executor from src module
orchestrator_executor, orchestrator_tools = create_orchestrator_executor(
    llm=llm,
    routing_executor=routing_executor,
    classification_executor=classification_executor,
)

print("Orchestrator tools:", [tool.name for tool in orchestrator_tools])
print("Agent ready.")

In [ ]:
# Run the orchestrator - watch the routing - classification coordination in the output

tweet = "$TSLA is finally breaking out after earnings! Loading more shares now  #bullish"

agentic_result = orchestrator_executor.invoke(
    {
        "input": (
            f"Classify this investor tweet: '{tweet}'. "
            "First choose the most appropriate classifier strategy, then give me the final label, "
            "confidence, and a short justification."
        )
    }
)

print("\n" + "-" * 62)
print("Answer:", agentic_result["output"])
print("-" * 62)


In [ ]:
print_trace(agentic_result)


### Automated Evaluation Through the Same Conversational Interface

The same agent can answer evaluation prompts as well, using the validation metrics tool instead of hardcoded notebook logic.


In [ ]:
evaluation_result = orchestrator_executor.invoke(
    {
        "input": (
            "Compare the available classifiers on the validation set and recommend which one we should deploy "
            "for noisy stock-market tweets with hashtags and cashtags."
        )
    }
)

print("\n" + "-" * 62)
print("Answer:", evaluation_result["output"])
print("-" * 62)


In [ ]:
print_trace(evaluation_result)


### Optional Memory for Follow-up Questions

We can also add short-term conversational memory so the user can ask follow-up questions about the previously recommended strategy.


In [ ]:
orchestrator_executor_mem = create_orchestrator_memory_executor(
    llm=llm,
    orchestrator_tools=orchestrator_tools,
)

print("Memory agent ready.")

In [ ]:
q1 = "For tweets full of cashtags and hashtags, which model strategy should we prefer- Remember your recommendation."
print(f"User  : {q1}")
ans1 = orchestrator_executor_mem.invoke({"input": q1})
print(f"Agent : {ans1['output']}")
print_trace(ans1)

print()

q2 = "Now classify this tweet using that recommendation: '$NVDA looks unstoppable right now #AI #stocks'"
print(f"User  : {q2}")
ans2 = orchestrator_executor_mem.invoke({"input": q2})
print(f"Agent : {ans2['output']}")
print_trace(ans2)
